# 📘 Notebook 4: Phase 4 Surrogate Analysis

This notebook summarizes the Phase 4 surrogate model outputs using the available evaluation report and training history.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "notebooks"))
import utils

root = Path.cwd()
phase4_dir = root /".."/ "outputs" / "phase4"
report_path = phase4_dir / "evaluation_report.json"
train_history_path = phase4_dir / "train_history.json"

print(f"Phase 4 output dir: {phase4_dir}")
if report_path.exists():
    with open(report_path, "r") as f:
        report = json.load(f)
    print("Loaded evaluation report")
    print(json.dumps(report, indent=2)[:2500])
else:
    print("evaluation_report.json not found")

if train_history_path.exists():
    with open(train_history_path, "r") as f:
        history = json.load(f)
    print("\nLoaded training history keys:", list(history.keys())[:20])
else:
    print("train_history.json not found")

In [ ]:
# Summarize the surrogate metrics in a table
if 'report' in locals():
    rows = []
    for target, metrics in report.get('overall', {}).items():
        rows.append({"target": target, "r2": metrics.get('r2'), "mae": metrics.get('mae')})
    metrics_df = pd.DataFrame(rows)
    display(metrics_df)

In [ ]:
# Plot per-seed surrogate accuracy if present
if 'report' in locals() and 'per_seed' in report:
    seed_df = pd.DataFrame([
        {"seed": seed, "n": info.get('n', None), "mae_v12": info.get('mae_v12'), "mae_v21": info.get('mae_v21')}
        for seed, info in report['per_seed'].items()
    ])
    seed_df = seed_df.sort_values('mae_v12')
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(seed_df['seed'], seed_df['mae_v12'], color='steelblue')
    ax.set_ylabel('MAE for v12')
    ax.set_title('Phase 4 surrogate MAE by seed')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()